In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
load_dotenv()
model = "openai/gpt-oss-20b"

In [52]:
from pydantic import BaseModel
from typing import Literal, Optional

class FitnessProfile(BaseModel):
    """Model representing a user's fitness profile with their goals, level, and constraints"""
    fitness_goal: Literal["BUILD MUSCLE", "LOSE FAT", "GENERAL FITNESS", "IMPROVE ENDURANCE"]
    level: Literal["BEGINNER", "INTERMEDIATE", "ADVANCED"]
    days_avl_per_week: int
    equipment_access: Literal["NO EQUIPMENT", "HOME DUMBBELLS", "FULL GYM"]
    limitations: Optional[str] = ""


In [53]:
# Create an Sampel Object for FitnessProfile
fitnessprofile = FitnessProfile(
    fitness_goal="BUILD MUSCLE",
    level="INTERMEDIATE",
    days_avl_per_week=4,
    equipment_access="HOME DUMBBELLS",
    limitations="None"
)
print(fitnessprofile)

fitness_goal='BUILD MUSCLE' level='INTERMEDIATE' days_avl_per_week=4 equipment_access='HOME DUMBBELLS' limitations='None'


In [54]:
# Generate Plan using LLM
# This function sends the inputs to the LLM and displays the result in a clearly formatted area
# The result is formatted as a weekly breakdown, day by day

def generate_plan(inputs: FitnessProfile) -> str:
    """
    Sends inputs to the LLM and generates a plan.
    
    Args:
        inputs: The input data to be sent to the LLM
        
    Returns:
        A formatted plan with weekly breakdown, day by day
    """
    # Send inputs to LLM
    try:
        llm_response = send_to_llm(inputs)
        return format_response(llm_response)
    except Exception as e:
        return "Something went wrong while generating the recommendations."        
   

In [55]:
def send_to_llm(inputs):
    """
    Sends the inputs to the LLM for processing.
    
    Args:
        inputs: The input data
        
    Returns:
        The LLM response
    """

    try:
        # Implementation to send inputs to LLM
        llm = ChatGroq(model=model,temperature=0)
        response = llm.invoke([
            SystemMessage(content="You are a personal fitness trainer. Use the provided tools to gather information and create a personalized workout plan."),
            HumanMessage(content=f"Create a workout plan based on this profile: {inputs}")
        ])
        return response.content
    except Exception as e:
        error_message = str(e).lower()
        print(f"LLM Error: {error_message}")
        raise Exception(error_message)



In [56]:
def format_response(response):
    """
    Formats the LLM response as a weekly, day-by-day breakdown.

    Args:
        response: Raw LLM response

    Returns:
        Formatted weekly breakdown
    """

    if not response:
        return "No weekly breakdown available."

    lines = response.strip().splitlines()

    formatted = ["## Weekly Breakdown"]

    current_day = None

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Detect day headings
        if line.lower().startswith("day "):
            current_day = line
            formatted.append(f"\n### {current_day}")
        else:
            formatted.append(f"- {line}")

    return "\n".join(formatted)
    


In [57]:
generate_plan(fitnessprofile)

'## Weekly Breakdown\n- ## 4‑Day Dumbbell‑Only Muscle‑Building Program\n- **Goal:** Hypertrophy (muscle growth)\n- **Level:** Intermediate (you’re comfortable with basic lifts and can handle moderate volume)\n- **Schedule:** 4 days per week (e.g., Mon, Tue, Thu, Fri or Mon, Wed, Fri, Sat)\n- **Equipment:** Dumbbells (you can use a pair of adjustable or a set that covers 20–60\u202flb each side).\n- **Limitations:** None – you can perform all movements safely.\n- > **Tip:** Keep a training log (date, weight, reps, notes). Aim to add 2–5\u202flb per set every 2–3 weeks when you hit the upper rep range comfortably.\n- ---\n- ### General Structure (per session)\n- | Phase | Time | Focus |\n- |-------|------|-------|\n- | Warm‑up | 5–10\u202fmin | Light cardio + dynamic mobility |\n- | Main sets | 45–60\u202fmin | 3–4 exercises per muscle group |\n- | Cool‑down | 5\u202fmin | Stretch + foam‑roll (optional) |\n- **Rest between sets:** 60–90\u202fsec for hypertrophy; 90–120\u202fsec for heavi